In [9]:
import pandas as pd
from google.cloud import storage
import os
import json
import subprocess
import tempfile


In [2]:
def list_blobs_with_prefix(bucket_name, prefix, file_extension='.tif', delimiter=None):
    

    storage_client = storage.Client(project='swhm-prod')

    # Note: Client.list_blobs requires at least package version 1.17.0.
    blobs = storage_client.list_blobs(bucket_name, prefix=prefix, delimiter=delimiter)

    # Note: The call returns a response only when the iterator is consumed.
    blob_list = []
    for blob in blobs:
        if blob.name.endswith(file_extension):
            blob_list.append(blob.name)

    if delimiter:
        print("Prefixes:")
        for prefix in blobs.prefixes:
            blob_list.append([prefix])
    
    return blob_list


In [3]:
def list_blobs_with_public_info(bucket_name, prefix, extension_filter=None):
    storage_client = storage.Client()
    blobs = storage_client.list_blobs(bucket_name, prefix=prefix)
    blob_info = []
    for blob in blobs:
        if not extension_filter or blob.name.endswith(extension_filter):
            public_url = f"https://storage.googleapis.com/{bucket_name}/{blob.name}"
            blob_info.append({
                'name': blob.name,
                'public_url': public_url
            })
    return blob_info

In [4]:
bucket_name = "live_data_layers"
storage_client = storage.Client()
blobs = storage_client.list_blobs(bucket_name, prefix="rasters/stac")
blobs

In [5]:
BUCKET_NAME = 'live_data_layers'
folder_name = 'stac'

In [6]:

blobsout = list_blobs_with_prefix(BUCKET_NAME,folder_name,".tif")

In [7]:
blob_info = list_blobs_with_public_info(BUCKET_NAME,folder_name,".tif")
blob_info


[{'name': 'stac/Age_of_Imperviousness/Age_of_Imperviousness.tif',
  'public_url': 'https://storage.googleapis.com/live_data_layers/stac/Age_of_Imperviousness/Age_of_Imperviousness.tif'},
 {'name': 'stac/Flow_Duration_Index/Flow_Duration_Index.tif',
  'public_url': 'https://storage.googleapis.com/live_data_layers/stac/Flow_Duration_Index/Flow_Duration_Index.tif'},
 {'name': 'stac/HSPF_Land_Cover_Type/HSPF_Land_Cover_Type.tif',
  'public_url': 'https://storage.googleapis.com/live_data_layers/stac/HSPF_Land_Cover_Type/HSPF_Land_Cover_Type.tif'},
 {'name': 'stac/Hydrologic_Response_Units/Hydrologic_Response_Units.tif',
  'public_url': 'https://storage.googleapis.com/live_data_layers/stac/Hydrologic_Response_Units/Hydrologic_Response_Units.tif'},
 {'name': 'stac/Imperviousness/Imperviousness.tif',
  'public_url': 'https://storage.googleapis.com/live_data_layers/stac/Imperviousness/Imperviousness.tif'},
 {'name': 'stac/Land_Cover/Land_Cover.tif',
  'public_url': 'https://storage.googleapis.c

In [7]:
blob = blob_info[0]
blob

{'name': 'stac/Age_of_Imperviousness/Age_of_Imperviousness.tif',
 'public_url': 'https://storage.googleapis.com/live_data_layers/stac/Age_of_Imperviousness/Age_of_Imperviousness.tif'}

In [ ]:
for blob in blob_info: 
    fn = blob['public_url']
    blob_path = blob['name']  # e.g., 'rasters/Age_of_Imperviousness.tif'
    base_name = os.path.splitext(os.path.basename(blob_path))[0]
    stac_path = f'../catalog-source/{base_name}/{base_name}.json'
    print(f'Checking {blob}...')
    os.makedirs(os.path.dirname(stac_path), exist_ok=True)
    r_cmd = f'rio stac {fn} --output {stac_path}'
    !{r_cmd}

    

Checking {'name': 'rasters/Age_of_Imperviousness.tif', 'public_url': 'https://storage.googleapis.com/live_data_layers/rasters/Age_of_Imperviousness.tif'}...
Checking {'name': 'rasters/Flow_Duration_Index.tif', 'public_url': 'https://storage.googleapis.com/live_data_layers/rasters/Flow_Duration_Index.tif'}...
Checking {'name': 'rasters/HSPF_Land_Cover_Type.tif', 'public_url': 'https://storage.googleapis.com/live_data_layers/rasters/HSPF_Land_Cover_Type.tif'}...
Checking {'name': 'rasters/Hydrologic_Response_Units.tif', 'public_url': 'https://storage.googleapis.com/live_data_layers/rasters/Hydrologic_Response_Units.tif'}...
Checking {'name': 'rasters/Imperviousness.tif', 'public_url': 'https://storage.googleapis.com/live_data_layers/rasters/Imperviousness.tif'}...
Checking {'name': 'rasters/Land_Cover.tif', 'public_url': 'https://storage.googleapis.com/live_data_layers/rasters/Land_Cover.tif'}...
Checking {'name': 'rasters/Land_Use.tif', 'public_url': 'https://storage.googleapis.com/live

In [12]:


for blob in blob_info:
    fn = blob["public_url"]
    blob_path = blob["name"]
    base = os.path.splitext(os.path.basename(blob_path))[0]

    stac_path = f"/tmp/{base}.json"
    os.makedirs(os.path.dirname(stac_path), exist_ok=True)

    # generate locally
    os.system(f"rio stac {fn} --output {stac_path}")

    # copy into the same GCS “folder”
    gcs_dest = f"gs://{BUCKET_NAME}/{os.path.dirname(blob_path)}/{base}.json"
    os.system(f"gsutil cp {stac_path} {gcs_dest}")
    print(f"Pushed to {gcs_dest}")

/Users/christiannilsen/.local/share/virtualenvs/data_pipelines-nRJIciM3/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Copying file:///tmp/Age_of_Imperviousness.json [Content-Type=application/json]...
/ [1 files][  1.6 KiB/  1.6 KiB]                                                
Operation completed over 1 objects/1.6 KiB.                                      


Pushed to gs://live_data_layers/stac/Age_of_Imperviousness/Age_of_Imperviousness.json


/Users/christiannilsen/.local/share/virtualenvs/data_pipelines-nRJIciM3/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Copying file:///tmp/Flow_Duration_Index.json [Content-Type=application/json]...
/ [1 files][  1.6 KiB/  1.6 KiB]                                                
Operation completed over 1 objects/1.6 KiB.                                      


Pushed to gs://live_data_layers/stac/Flow_Duration_Index/Flow_Duration_Index.json


/Users/christiannilsen/.local/share/virtualenvs/data_pipelines-nRJIciM3/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Copying file:///tmp/HSPF_Land_Cover_Type.json [Content-Type=application/json]...
/ [1 files][  1.6 KiB/  1.6 KiB]                                                
Operation completed over 1 objects/1.6 KiB.                                      


...

Pushed to gs://live_data_layers/stac/Total_Zinc_Concentration/Total_Zinc_Concentration.json


/Users/christiannilsen/.local/share/virtualenvs/data_pipelines-nRJIciM3/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Copying file:///tmp/Traffic.json [Content-Type=application/json]...
/ [1 files][  1.6 KiB/  1.6 KiB]                                                
Operation completed over 1 objects/1.6 KiB.                                      


Pushed to gs://live_data_layers/stac/Traffic/Traffic.json


/Users/christiannilsen/.local/share/virtualenvs/data_pipelines-nRJIciM3/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Copying file:///tmp/copper_concentration_ug_per_L.json [Content-Type=application/json]...
/ [1 files][  1.7 KiB/  1.7 KiB]                                                
Operation completed over 1 objects/1.7 KiB.                                      


Pushed to gs://live_data_layers/stac/copper_concentration_ug_per_L/copper_concentration_ug_per_L.json
